# Download GGUF from HuggingFace → SRD .axm

Downloads pre-quantized GGUF models directly from HuggingFace and packs
them into signed `.axm` governance containers.

**No safetensors needed. No quantization step. Works on CPU or GPU.**

Steps:
1. `Cell 1` — pip install
2. `Cell 2` — Clone axiom + AXIOM_MASTER_KEY
3. `Cell 3` — Configure model IDs ← **edit here**
4. `Cell 4` — Download GGUFs from HuggingFace
5. `Cell 5` — Pack each GGUF → signed .axm
6. `Cell 6` — Verify HMAC proofs
7. `Cell 7` — Summary + llama.cpp inference commands


In [ ]:
# Cell 1 — install dependencies
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "huggingface_hub", "tqdm"], check=True)
print("✓ ready")


In [ ]:
# Cell 2 — clone axiom + set AXIOM_MASTER_KEY
import os, secrets, subprocess, sys
from pathlib import Path

AXIOM_DIR  = Path("/content/axiom")       # RunPod: /workspace/axiom
OUTPUT_DIR = Path("/content/axm_output")  # RunPod: /workspace/axm_output
BRANCH     = "claude/srd-multimodal"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not AXIOM_DIR.is_dir():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1",
        "https://github.com/orivael-dev/axiom.git", str(AXIOM_DIR)], check=True)
    print("✓ axiom cloned")
else:
    subprocess.run(["git", "-C", str(AXIOM_DIR), "pull"], check=True)
    print(f"✓ axiom at {AXIOM_DIR}")
sys.path.insert(0, str(AXIOM_DIR))

KEY_FILE = OUTPUT_DIR / "axiom_master.key"
if os.environ.get("AXIOM_MASTER_KEY"):
    print("AXIOM_MASTER_KEY: from environment")
elif KEY_FILE.is_file():
    os.environ["AXIOM_MASTER_KEY"] = KEY_FILE.read_text().strip()
    print(f"AXIOM_MASTER_KEY: restored from {KEY_FILE}")
else:
    key = secrets.token_hex(32)
    os.environ["AXIOM_MASTER_KEY"] = key
    KEY_FILE.write_text(key)
    print(f"AXIOM_MASTER_KEY: generated → {KEY_FILE}")
    print("  ⚠ back this file up to verify AXMs later")

In [ ]:
# Cell 3 — ⚙️  CONFIGURE MODELS
#
# repo_id  — HuggingFace repo containing GGUF files
# filename — exact filename inside that repo (Files tab on HF)
# hf_model — base model ID for AXM metadata
#
# These are the SRD lab collection — baseline Q4_K_M GGUFs from HF.
# To pack your own SRD-corrected GGUF, point gguf_path at it in Cell 4.
MODELS = {
    "smollm2-135m": {
        "repo_id":  "HuggingFaceTB/SmolLM2-135M-Instruct-GGUF",
        "filename": "smollm2-135m-instruct-q4_k_m.gguf",
        "hf_model": "HuggingFaceTB/SmolLM2-135M-Instruct",
    },
    "qwen25-0p5b": {
        "repo_id":  "Qwen/Qwen2.5-Coder-0.5B-Instruct-GGUF",
        "filename": "qwen2.5-coder-0.5b-instruct-q4_k_m.gguf",
        "hf_model": "Qwen/Qwen2.5-Coder-0.5B-Instruct",
    },
    "gemma3-1b": {
        "repo_id":  "bartowski/gemma-3-1b-it-GGUF",
        "filename": "gemma-3-1b-it-Q4_K_M.gguf",
        "hf_model": "google/gemma-3-1b-it",
    },
    "tinyllama-1b": {
        "repo_id":  "TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF",
        "filename": "tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf",
        "hf_model": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    },
    "deepseek-r1-1b5": {
        "repo_id":  "bartowski/DeepSeek-R1-Distill-Qwen-1.5B-GGUF",
        "filename": "DeepSeek-R1-Distill-Qwen-1.5B-Q4_K_M.gguf",
        "hf_model": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    },
    "llama32-1b": {
        "repo_id":  "bartowski/Llama-3.2-1B-Instruct-GGUF",
        "filename": "Llama-3.2-1B-Instruct-Q4_K_M.gguf",
        "hf_model": "meta-llama/Llama-3.2-1B-Instruct",
    },
    "qwen3-1b7": {
        "repo_id":  "Qwen/Qwen3-1.7B-GGUF",
        "filename": "Qwen3-1.7B-Q4_K_M.gguf",
        "hf_model": "Qwen/Qwen3-1.7B",
    },
    "phi-2": {
        "repo_id":  "TheBloke/phi-2-GGUF",
        "filename": "phi-2.Q4_K_M.gguf",
        "hf_model": "microsoft/phi-2",
    },
}

# Comment out any models you don't want to pack:
# del MODELS["phi-2"]

# HF token — needed for gated models (Llama-3.2 requires accepting license)
HF_TOKEN = os.environ.get("HF_TOKEN", "")   # or paste: HF_TOKEN = "hf_..."

print("Models configured:")
for name, cfg in MODELS.items():
    print(f"  {name:18s}  {cfg['repo_id']}/{cfg['filename']}")
if not HF_TOKEN:
    print("\n⚠ HF_TOKEN not set — llama32-1b (gated) will fail without it.")
    print("  Set it: HF_TOKEN = 'hf_...'")

In [ ]:
# Cell 4 — Download GGUFs from HuggingFace
import os, time
from pathlib import Path
from huggingface_hub import hf_hub_download

GGUF_DIR = OUTPUT_DIR / "gguf"
GGUF_DIR.mkdir(exist_ok=True)
RESULTS = {}

for name, cfg in MODELS.items():
    dest = GGUF_DIR / cfg["filename"]
    if dest.exists():
        gb = dest.stat().st_size / 1024**3
        print(f"  ✓ {name:15s}  already downloaded  ({gb:.2f} GB)")
        RESULTS[name] = {"gguf_path": str(dest), "status": "ready", **cfg}
        continue

    print(f"  Downloading {name}: {cfg['repo_id']}/{cfg['filename']} ...")
    t0 = time.time()
    try:
        path = hf_hub_download(
            repo_id   = cfg["repo_id"],
            filename  = cfg["filename"],
            local_dir = str(GGUF_DIR),
            token     = HF_TOKEN or None,
        )
        elapsed = time.time() - t0
        gb = Path(path).stat().st_size / 1024**3
        print(f"  ✓ {name:15s}  {gb:.2f} GB  {elapsed/60:.1f} min")
        RESULTS[name] = {"gguf_path": path, "status": "ready", **cfg}
    except Exception as e:
        print(f"  ✗ {name:15s}  FAILED: {e}")
        print(f"     Check repo_id/filename are correct and HF_TOKEN is set for gated models.")
        RESULTS[name] = {"gguf_path": None, "status": "failed", **cfg}

print("\nDownloads complete.")


In [ ]:
# Cell 5 — Pack each GGUF into a signed .axm container
import json, subprocess, sys, time
from pathlib import Path

PACK_SCRIPT = AXIOM_DIR / "research" / "quant" / "pack_gguf_to_axm.py"

for name, r in RESULTS.items():
    if not r.get("gguf_path"):
        print(f"  SKIP  {name}  (download failed)")
        continue

    axm_out   = OUTPUT_DIR / f"{name}.axm"
    stats_out = OUTPUT_DIR / f"{name}_pack_stats.json"

    if axm_out.exists():
        gb = axm_out.stat().st_size / 1024**3
        print(f"  ✓ {name:15s}  already packed  ({gb:.3f} GB)  -- re-run to repack")
        r["axm_path"] = str(axm_out)
        continue

    print(f"\n{'─'*50}")
    print(f"Packing: {name}")
    t0 = time.time()
    result = subprocess.run(
        [
            sys.executable, str(PACK_SCRIPT),
            "--gguf",       r["gguf_path"],
            "--output",     str(axm_out),
            "--model",      r["hf_model"],
            "--stats-json", str(stats_out),
        ],
        cwd=str(AXIOM_DIR),
        capture_output=True, text=True,
    )
    elapsed = time.time() - t0

    if result.stdout: print(result.stdout)
    if result.returncode != 0:
        print(f"  ✗ FAILED (rc={result.returncode})")
        for ln in result.stderr.strip().splitlines()[-15:]:
            print(f"  {ln}")
        r["axm_path"] = None
        continue

    gb = axm_out.stat().st_size / 1024**3
    stats = json.loads(stats_out.read_text()) if stats_out.exists() else {}
    r["axm_path"]    = str(axm_out)
    r["fingerprint"] = stats.get("fingerprint")
    r["axm_gb"]      = round(gb, 3)
    r["pack_s"]      = round(elapsed, 1)
    print(f"  ✓ {axm_out.name}  ({gb:.3f} GB)  {elapsed:.1f}s")


In [ ]:
# Cell 6 — Verify HMAC proof chains
import json, subprocess, sys

AXM_CLI = AXIOM_DIR / "axm_cli.py"

for name, r in RESULTS.items():
    if not r.get("axm_path"):
        print(f"  SKIP  {name}")
        continue
    out = subprocess.run(
        [sys.executable, str(AXM_CLI), "verify", r["axm_path"]],
        cwd=str(AXIOM_DIR), capture_output=True, text=True,
    )
    try:
        data = json.loads(out.stdout)
    except Exception:
        data = {"verified": False, "error": out.stdout + out.stderr}
    ok = data.get("verified", False)
    fp = data.get("fingerprint", "?")
    proofs = data.get("proofs_checked", "?")
    mark = "✓" if ok else "✗"
    print(f"  {mark} [{name:15s}]  fingerprint={fp}  proofs={proofs}")
    r["verified"]    = ok
    r["fingerprint"] = fp
    r["proofs"]      = proofs
    if not ok:
        print(f"     {data.get('error', 'unknown error')}")


In [ ]:
# Cell 7 — Summary
import json
from pathlib import Path

print("\n" + "="*70)
print("RESULTS")
print("="*70)
print(f"{'Model':<15} {'Status':<6} {'AXM GB':>7} {'Fingerprint':<12} {'Proofs':>6}")
print("-"*70)
for name, r in RESULTS.items():
    ok  = "✓" if r.get("verified") else ("packed" if r.get("axm_path") else "✗")
    gb  = r.get("axm_gb", 0) or 0
    fp  = r.get("fingerprint", "?") or "?"
    pr  = str(r.get("proofs", "?") or "?")
    print(f"{name:<15} {ok:<6} {gb:>6.3f}G {fp:<12} {pr:>6}")
print("="*70)

# Save results
out = OUTPUT_DIR / "results.json"
out.write_text(json.dumps(RESULTS, indent=2))
print(f"\nResults saved: {out}")

# Artifacts
print("\n--- Artifacts ---")
for name, r in RESULTS.items():
    if r.get("axm_path"):
        print(f"  .axm   {r['axm_path']}")
    if r.get("gguf_path"):
        print(f"  .gguf  {r['gguf_path']}")

# Inference commands
LLAMACPP = Path("/content/llama.cpp")
print("\n--- llama.cpp inference ---")
for name, r in RESULTS.items():
    if r.get("gguf_path"):
        print(f"  # {name}")
        print(f"  {LLAMACPP}/build/bin/llama-cli -m '{r['gguf_path']}' --ngl 99 -p 'Hello' -n 64")
